In [ ]:
# config da camada gold
# centraliza fontes e destinos para manter o modelo dimensional fácil de revisar

try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

# separa as tabelas tratadas da silver das estruturas analíticas publicadas na gold
SCHEMA_ORIGEM = "silver"
SCHEMA_DESTINO = "gold"
# reúne os nomes das fontes para evitar referências diferentes à mesma tabela
TABELA_FILMES_ORIGEM = "tb_info_filmes"
TABELA_GENEROS_ORIGEM = "tb_generos"
TABELA_PESSOAS_ORIGEM = "tb_pessoas_empresas"
TABELA_AVALIACOES_ORIGEM = "tb_avaliacoes_usuarios"

# reúne os nomes das entregas dimensionais e da tabela destinada ao rag
TABELA_DIM_FILMES = "dim_movies"
TABELA_DIM_GENEROS = "dim_genres"
TABELA_DIM_PESSOAS = "dim_people"
TABELA_DIM_PRODUTORAS = "dim_companies"
TABELA_DIM_AVALIACOES = "dim_reviews"
TABELA_FINANCEIRO_ORIGEM = "tb_financeiro_filmes"
TABELA_METRICAS_ORIGEM = "tb_metricas_engajamento"
TABELA_FATO_PERFORMANCE = "fact_movies_performance"
TABELA_PONTE_GENEROS = "bridge_movie_genre"
TABELA_PONTE_PESSOAS = "bridge_movie_person"
TABELA_PONTE_PRODUTORAS = "bridge_movie_company"
TABELA_CONTEXTO_GENAI = "gold_genai_movies_context"

# cria o schema sem substituir tabelas antes da reconstrução de cada saída
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

# usa sha2 e reserva 60 bits para que a chave caiba em bigint sem perder estabilidade
def gerar_chave_substituta(*campos):
    conteudo = F.to_json(F.struct(*[
        campo.cast("string").alias(f"campo_{indice}")
        for indice, campo in enumerate(campos)
    ]))
    return F.conv(F.substring(F.sha2(conteudo, 256), 1, 15), 16, 10).cast("BIGINT")


# confere chaves antes de joins ou gravações para evitar multiplicação de registros
def validar_chaves(df, chaves, nome):
    chave_vazia = F.lit(False)
    for chave in chaves:
        chave_vazia = chave_vazia | F.col(chave).isNull() | (F.trim(F.col(chave).cast("string")) == "")
    if df.where(chave_vazia).limit(1).count():
        raise AssertionError(f"{nome}: chave obrigatória vazia em {chaves}")
    if df.groupBy(*chaves).count().where(F.col("count") > 1).limit(1).count():
        raise AssertionError(f"{nome}: chave duplicada ou colisão de hash em {chaves}")


# compara os conjuntos de chaves para encontrar perdas e registros inesperados
def validar_cobertura(esperado, obtido, chaves, nome):
    esperado = esperado.select(*chaves).distinct()
    obtido = obtido.select(*chaves).distinct()
    faltantes = esperado.join(obtido, chaves, "left_anti")
    extras = obtido.join(esperado, chaves, "left_anti")
    if faltantes.limit(1).count() or extras.limit(1).count():
        display(faltantes.limit(20))
        display(extras.limit(20))
        raise AssertionError(f"{nome}: cobertura divergente nas chaves {chaves}")


In [ ]:
# tabela gold.dim_movies
# organiza os atributos descritivos no grão de uma linha por id natural da origem
# mantém o catálogo completo porque somente a fato precisa restringir filmes não lançados
# preserva a chave canônica como apoio às análises que não podem contar cópias da mesma obra

# carrega uma vez as fontes compartilhadas pelas dimensões e relacionamentos
df_filmes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_FILMES_ORIGEM}")
df_generos_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_GENEROS_ORIGEM}")
df_pessoas_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_PESSOAS_ORIGEM}")
df_avaliacoes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_AVALIACOES_ORIGEM}")

# interrompe a execução cedo se a silver não entregar algum atributo obrigatório
colunas_filmes_esperadas = {
    "id_filme", "id_obra_canonica", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status_filme", "sinopse", "frase_divulgacao"
}
colunas_filmes_ausentes = colunas_filmes_esperadas.difference(df_filmes_silver.columns)
if colunas_filmes_ausentes:
    raise ValueError(f"Colunas ausentes na Silver de filmes: {sorted(colunas_filmes_ausentes)}")

validar_chaves(df_filmes_silver, ["id_filme"], "silver.tb_info_filmes")

# gera uma chave estável pelo id natural para o registro manter a identidade em reprocessamentos
df_dim_movies = (
    df_filmes_silver
    .where(F.col("id_filme").isNotNull() & (F.length(F.trim(F.col("id_filme"))) > 0))
    .withColumn("sk_movie_id", gerar_chave_substituta(F.lit("movie"), F.col("id_filme").cast("string")))
    .select(
        "sk_movie_id", "id_filme", "id_obra_canonica", "titulo",
        "titulo_original", "idioma_original", "data_lancamento",
        "ano_lancamento", "duracao_minutos", "status_filme",
        "sinopse", "frase_divulgacao"
    )
)

validar_chaves(df_dim_movies, ["sk_movie_id"], "gold.dim_movies")

(df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
display(df_dim_movies.limit(10))

In [ ]:
# tabela gold.dim_genres
# cria um catálogo único de gêneros para evitar repetição dos nomes em cada filme

# usa uma versão normalizada apenas para comparar e preserva um nome legível na dimensão
df_dim_genres = (
    df_generos_silver
    .select(F.trim(F.col("nome_genero")).alias("nome_genero"))
    .where(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .withColumn("nome_genero_normalizado", F.lower(F.col("nome_genero")))
    .groupBy("nome_genero_normalizado")
    .agg(F.min("nome_genero").alias("nome_genero"))
    .withColumn("sk_genre_id", gerar_chave_substituta(F.lit("genre"), F.col("nome_genero_normalizado")))
    .select("sk_genre_id", "nome_genero")
)

validar_chaves(df_dim_genres, ["sk_genre_id"], "gold.dim_genres")

(df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")
display(df_dim_genres.orderBy("nome_genero").limit(10))

In [ ]:
# tabelas gold.dim_people e gold.dim_companies
# separa pessoas físicas de empresas para respeitar as dimensões pedidas no modelo

# remove nomes vazios e resíduos antes de gerar as chaves substitutas
df_entidades = (
    df_pessoas_silver
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_entidade"),
        F.trim(F.col("tipo_entidade")).alias("tipo_entidade")
    )
    .where(F.col("nome_entidade").isNotNull() & (F.col("nome_entidade") != ""))
    .where(~F.col("nome_entidade").rlike(r"^\d+$"))
)

# inclui o tipo na chave porque um nome pode aparecer como ator, diretor ou roteirista
df_dim_people = (
    df_entidades
    .where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .dropDuplicates(["nome_entidade", "tipo_entidade"])
    .withColumn("sk_person_id", gerar_chave_substituta(F.lit("person"), F.col("nome_entidade"), F.col("tipo_entidade")))
    .select("sk_person_id", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)

# separa as produtoras das pessoas físicas
df_dim_companies = (
    df_entidades
    .where(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", gerar_chave_substituta(F.lit("company"), F.col("nome_produtora")))
    .select("sk_company_id", "nome_produtora")
)

validar_chaves(df_dim_people, ["sk_person_id"], "gold.dim_people")
validar_chaves(df_dim_companies, ["sk_company_id"], "gold.dim_companies")

# reconstrói as duas dimensões em delta para que o resultado seja idempotente
for df_dimensao, tabela in [
    (df_dim_people, TABELA_DIM_PESSOAS),
    (df_dim_companies, TABELA_DIM_PRODUTORAS),
]:
    (df_dimensao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SCHEMA_DESTINO}.{tabela}"
    ))
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")

display(df_dim_people.limit(10))
display(df_dim_companies.limit(10))

In [ ]:
# tabela gold.dim_reviews
# transforma avaliações individuais nas duas métricas resumidas exigidas por filme

# mantém todas as avaliações na contagem e deixa o spark ignorar notas nulas somente na média
df_reviews_agregadas = (
    df_avaliacoes_silver
    .where(F.col("id_filme").isNotNull())
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("INT").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("DOUBLE").alias("nota_media_usuarios")
    )
)

# troca a chave natural pela surrogate key usada em todo o modelo gold
df_dim_reviews = (
    df_reviews_agregadas
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .withColumn("sk_review_id", gerar_chave_substituta(F.lit("review"), F.col("id_filme").cast("string")))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

validar_chaves(df_dim_reviews, ["sk_review_id"], "gold.dim_reviews")

(df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}")
display(df_dim_reviews.limit(10))

In [ ]:
# tabela gold.fact_movies_performance
# centraliza finanças e engajamento no grão obrigatório de uma linha por filme lançado

# consome somente métricas já tipadas e validadas para evitar regras de limpeza na gold
df_financeiro_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_FINANCEIRO_ORIGEM}")
df_metricas_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_METRICAS_ORIGEM}")

# confirma o contrato das duas fontes antes dos joins que formam a fato
colunas_financeiro_esperadas = {
    "id_filme", "orcamento_usd", "receita_usd",
    "cotacao_dolar_brl", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual"
}
colunas_metricas_esperadas = {
    "id_filme", "popularidade", "nota_media_tmdb",
    "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
}
for colunas_esperadas, df_origem, nome_origem in [
    (colunas_financeiro_esperadas, df_financeiro_silver, "financeira"),
    (colunas_metricas_esperadas, df_metricas_silver, "métricas"),
]:
    colunas_ausentes = colunas_esperadas.difference(df_origem.columns)
    if colunas_ausentes:
        raise ValueError(f"Colunas ausentes na Silver {nome_origem}: {sorted(colunas_ausentes)}")

validar_chaves(df_financeiro_silver, ["id_filme"], "silver.tb_financeiro_filmes")
validar_chaves(df_metricas_silver, ["id_filme"], "silver.tb_metricas_engajamento")

# usa left join para não perder filmes lançados que possuem alguma métrica opcional ausente
df_fact_movies_performance = (
    df_dim_movies.where(F.col("status_filme") == "Lançado").select("id_filme", "sk_movie_id")
    .join(df_financeiro_silver, on="id_filme", how="left")
    .join(df_metricas_silver, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "cotacao_dolar_brl",
        "orcamento_brl", "receita_brl", "lucro_usd",
        "lucro_brl", "margem_lucro_percentual",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    )
)

validar_chaves(df_fact_movies_performance, ["sk_movie_id"], "gold.fact_movies_performance")

(df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_FATO_PERFORMANCE}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_FATO_PERFORMANCE}")
display(df_fact_movies_performance.limit(10))

In [ ]:
# tabelas gold.bridge_movie_genre, gold.bridge_movie_person e gold.bridge_movie_company
# representa relações muitos para muitos sem multiplicar as linhas da tabela fato

# cada ponte conserva somente pares únicos de chaves substitutas
df_bridge_movie_genre = (
    df_generos_silver
    .select("id_filme", F.lower(F.trim(F.col("nome_genero"))).alias("nome_genero_normalizado"))
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(
        df_dim_genres.withColumn("nome_genero_normalizado", F.lower(F.trim(F.col("nome_genero")))),
        on="nome_genero_normalizado", how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

# liga atores, diretores e roteiristas à dimensão de pessoas pelo nome e pelo tipo
df_bridge_movie_person = (
    df_pessoas_silver
    .where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(
        df_dim_people,
        (F.trim(F.col("nome_entidade")) == F.col("nome_pessoa")) &
        (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)

# liga somente entidades classificadas como produtora à dimensão de empresas
df_bridge_movie_company = (
    df_pessoas_silver.where(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(df_dim_companies, F.trim(F.col("nome_entidade")) == F.col("nome_produtora"), how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)

# reconstrói as três pontes depois que todos os pares repetidos foram removidos
for df_ponte, tabela in [
    (df_bridge_movie_genre, TABELA_PONTE_GENEROS),
    (df_bridge_movie_person, TABELA_PONTE_PESSOAS),
    (df_bridge_movie_company, TABELA_PONTE_PRODUTORAS),
]:
    (df_ponte.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SCHEMA_DESTINO}.{tabela}"
    ))
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")

display(df_bridge_movie_genre.limit(10))
display(df_bridge_movie_person.limit(10))
display(df_bridge_movie_company.limit(10))

In [ ]:
# tabela gold.gold_genai_movies_context
# reúne as informações em frase corrida e mantém o documento mesmo com campos ausentes

# a fonte não informa protagonismo, então agrega os atores disponíveis sem inventar uma ordem
df_pessoas_contexto = (
    df_bridge_movie_person
    .join(df_dim_people, "sk_person_id", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_set(
            F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa"))
        ))).alias("atores"),
        F.concat_ws(", ", F.sort_array(F.collect_set(
            F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa"))
        ))).alias("diretores"),
    )
)


# substitui campos vazios antes da concatenação para não perder o texto inteiro
def valor_ou_fallback(nome_coluna, fallback):
    valor = F.trim(F.coalesce(F.col(nome_coluna).cast("string"), F.lit("")))
    return F.when(valor != "", valor).otherwise(F.lit(fallback))


# identifica a moeda somente quando existe um valor financeiro
def valor_em_dolares(nome_coluna, fallback):
    return F.when(
        F.col(nome_coluna).isNotNull(),
        F.concat(F.lit("US$ "), F.col(nome_coluna).cast("string")),
    ).otherwise(F.lit(fallback))


# os left joins preservam também filmes sem elenco, sinopse ou métricas financeiras
df_contexto_genai = (
    df_dim_movies
    .select(
        "sk_movie_id", F.col("id_filme").alias("movie_id"),
        valor_ou_fallback("titulo", "Não informado").alias("title"),
        "ano_lancamento", "sinopse",
    )
    .join(df_pessoas_contexto, "sk_movie_id", "left")
    .join(
        df_fact_movies_performance.select("sk_movie_id", "receita_usd", "orcamento_usd"),
        "sk_movie_id", "left",
    )
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("title"),
            F.lit(", lançado no ano de "), valor_ou_fallback("ano_lancamento", "ano não informado"),
            F.lit(", faturou "), valor_em_dolares("receita_usd", "valor não informado"),
            F.lit(" e teve um custo de "), valor_em_dolares("orcamento_usd", "valor não informado"),
            F.lit(". Estrelado por "), valor_ou_fallback("atores", "elenco não informado"),
            F.lit(" e dirigido por "), valor_ou_fallback("diretores", "diretor não informado"),
            F.lit(", o filme possui a seguinte sinopse: "),
            valor_ou_fallback("sinopse", "sinopse não informada"), F.lit("."),
        ),
    )
    .select("movie_id", "title", "llm_context_document")
)

validar_chaves(df_contexto_genai, ["movie_id"], "gold.gold_genai_movies_context")
validar_cobertura(
    df_dim_movies.select(F.col("id_filme").alias("movie_id")),
    df_contexto_genai, ["movie_id"], "gold.gold_genai_movies_context",
)
(df_contexto_genai.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{SCHEMA_DESTINO}.{TABELA_CONTEXTO_GENAI}"))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_CONTEXTO_GENAI}")
display(df_contexto_genai.limit(10))


In [ ]:
# validação do modelo gold
# confere tipos, chaves e cobertura das saídas, além de mostrar vínculos ausentes na origem

# relê todas as tabelas para conferir o resultado efetivamente gravado
tabelas_gold = {
    nome: spark.table(f"{SCHEMA_DESTINO}.{nome}")
    for nome in [
        TABELA_DIM_FILMES, TABELA_DIM_GENEROS, TABELA_DIM_PESSOAS,
        TABELA_DIM_PRODUTORAS, TABELA_DIM_AVALIACOES, TABELA_FATO_PERFORMANCE,
        TABELA_PONTE_GENEROS, TABELA_PONTE_PESSOAS, TABELA_PONTE_PRODUTORAS,
        TABELA_CONTEXTO_GENAI,
    ]
}
dm = tabelas_gold[TABELA_DIM_FILMES]
dg = tabelas_gold[TABELA_DIM_GENEROS]
dp = tabelas_gold[TABELA_DIM_PESSOAS]
dc = tabelas_gold[TABELA_DIM_PRODUTORAS]
dr = tabelas_gold[TABELA_DIM_AVALIACOES]
fato = tabelas_gold[TABELA_FATO_PERFORMANCE]
bg = tabelas_gold[TABELA_PONTE_GENEROS]
bp = tabelas_gold[TABELA_PONTE_PESSOAS]
bc = tabelas_gold[TABELA_PONTE_PRODUTORAS]
contexto = tabelas_gold[TABELA_CONTEXTO_GENAI]

# cobre o contrato de todas as tabelas exigidas pelo pdf
contratos = [
    (TABELA_DIM_FILMES, ["sk_movie_id"], {
        "sk_movie_id": "bigint", "id_filme": "string", "titulo": "string",
        "titulo_original": "string", "data_lancamento": "date", "ano_lancamento": "int",
        "duracao_minutos": "int", "idioma_original": "string", "status_filme": "string",
        "sinopse": "string", "frase_divulgacao": "string",
    }),
    (TABELA_DIM_GENEROS, ["sk_genre_id"], {"sk_genre_id": "bigint", "nome_genero": "string"}),
    (TABELA_DIM_PESSOAS, ["sk_person_id"], {
        "sk_person_id": "bigint", "nome_pessoa": "string", "tipo_pessoa": "string",
    }),
    (TABELA_DIM_PRODUTORAS, ["sk_company_id"], {"sk_company_id": "bigint", "nome_produtora": "string"}),
    (TABELA_DIM_AVALIACOES, ["sk_review_id"], {
        "sk_review_id": "bigint", "sk_movie_id": "bigint",
        "qtd_avaliacoes_usuarios": "int", "nota_media_usuarios": "double",
    }),
    (TABELA_FATO_PERFORMANCE, ["sk_movie_id"], {
        "sk_movie_id": "bigint", "orcamento_usd": "decimal(18,2)", "receita_usd": "decimal(18,2)",
        "lucro_usd": "decimal(18,2)", "orcamento_brl": "decimal(18,2)", "receita_brl": "decimal(18,2)",
        "lucro_brl": "decimal(18,2)", "cotacao_dolar_brl": "decimal(12,6)",
        "margem_lucro_percentual": "decimal(10,2)", "popularidade": "double",
        "nota_media_tmdb": "double", "qtd_votos_tmdb": "int",
        "nota_media_imdb": "double", "qtd_votos_imdb": "int",
    }),
    (TABELA_PONTE_GENEROS, ["sk_movie_id", "sk_genre_id"], {"sk_movie_id": "bigint", "sk_genre_id": "bigint"}),
    (TABELA_PONTE_PESSOAS, ["sk_movie_id", "sk_person_id"], {"sk_movie_id": "bigint", "sk_person_id": "bigint"}),
    (TABELA_PONTE_PRODUTORAS, ["sk_movie_id", "sk_company_id"], {"sk_movie_id": "bigint", "sk_company_id": "bigint"}),
    (TABELA_CONTEXTO_GENAI, ["movie_id"], {"movie_id": "string", "title": "string", "llm_context_document": "string"}),
]
for nome, chaves, tipos in contratos:
    df = tabelas_gold[nome]
    tipos_obtidos = dict(df.dtypes)
    divergencias = [
        f"{coluna}: esperado {tipo}, obtido {tipos_obtidos.get(coluna)}"
        for coluna, tipo in tipos.items() if tipos_obtidos.get(coluna) != tipo
    ]
    if divergencias:
        raise AssertionError(f"gold.{nome}: " + "; ".join(divergencias))
    validar_chaves(df, chaves, f"gold.{nome}")

# valida também as chaves de negócio para impedir dimensões duplicadas com sk diferentes
for df, chaves, nome in [
    (dm, ["id_filme"], TABELA_DIM_FILMES),
    (dg, ["nome_genero"], TABELA_DIM_GENEROS),
    (dp, ["nome_pessoa", "tipo_pessoa"], TABELA_DIM_PESSOAS),
    (dc, ["nome_produtora"], TABELA_DIM_PRODUTORAS),
    (dr, ["sk_movie_id"], TABELA_DIM_AVALIACOES),
]:
    validar_chaves(df, chaves, f"gold.{nome}")

# procura relações órfãs nos dois lados das pontes e nas avaliações
for filha, pai, chave, nome in [
    (fato, dm, "sk_movie_id", TABELA_FATO_PERFORMANCE),
    (dr, dm, "sk_movie_id", TABELA_DIM_AVALIACOES),
    (bg, dm, "sk_movie_id", TABELA_PONTE_GENEROS),
    (bg, dg, "sk_genre_id", TABELA_PONTE_GENEROS),
    (bp, dm, "sk_movie_id", TABELA_PONTE_PESSOAS),
    (bp, dp, "sk_person_id", TABELA_PONTE_PESSOAS),
    (bc, dm, "sk_movie_id", TABELA_PONTE_PRODUTORAS),
    (bc, dc, "sk_company_id", TABELA_PONTE_PRODUTORAS),
]:
    if filha.join(pai.select(chave), chave, "left_anti").limit(1).count():
        raise AssertionError(f"gold.{nome}: relacionamento órfão em {chave}")

validar_cobertura(df_filmes_silver, dm, ["id_filme"], TABELA_DIM_FILMES)
validar_cobertura(
    dm.where(F.col("status_filme") == "Lançado"), fato,
    ["sk_movie_id"], TABELA_FATO_PERFORMANCE,
)
validar_cobertura(
    dm.select(F.col("id_filme").alias("movie_id")), contexto,
    ["movie_id"], TABELA_CONTEXTO_GENAI,
)

# registra explicitamente dados da silver sem metadados de filme, antes de conferir a cobertura dos joins
auditoria_origem = []
for nome, origem in [
    (TABELA_AVALIACOES_ORIGEM, df_avaliacoes_silver),
    (TABELA_GENEROS_ORIGEM, df_generos_silver),
    (TABELA_PESSOAS_ORIGEM, df_pessoas_silver),
    (TABELA_FINANCEIRO_ORIGEM, df_financeiro_silver),
    (TABELA_METRICAS_ORIGEM, df_metricas_silver),
]:
    sem_filme = origem.join(dm.select("id_filme"), "id_filme", "left_anti")
    quantidade = sem_filme.count()
    auditoria_origem.append((nome, quantidade))
    if quantidade:
        print(f"Aviso: silver.{nome} possui {quantidade} registros sem filme correspondente")
        display(sem_filme.limit(10))
display(spark.createDataFrame(auditoria_origem, ["tabela_silver", "registros_sem_filme"]))

# os vínculos com filme conhecido precisam sobreviver integralmente aos joins
filmes_conhecidos = dm.select("id_filme")
validar_cobertura(
    df_reviews_agregadas.join(filmes_conhecidos, "id_filme", "left_semi"),
    dr.join(dm.select("sk_movie_id", "id_filme"), "sk_movie_id"),
    ["id_filme"], TABELA_DIM_AVALIACOES,
)
validar_cobertura(
    df_generos_silver.join(filmes_conhecidos, "id_filme", "left_semi"),
    bg.join(dm.select("sk_movie_id", "id_filme"), "sk_movie_id").join(dg, "sk_genre_id"),
    ["id_filme", "nome_genero"], TABELA_PONTE_GENEROS,
)
validar_cobertura(
    df_pessoas_silver.where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(filmes_conhecidos, "id_filme", "left_semi"),
    bp.join(dm.select("sk_movie_id", "id_filme"), "sk_movie_id").join(dp, "sk_person_id")
    .select("id_filme", F.col("nome_pessoa").alias("nome_entidade"), F.col("tipo_pessoa").alias("tipo_entidade")),
    ["id_filme", "nome_entidade", "tipo_entidade"], TABELA_PONTE_PESSOAS,
)
validar_cobertura(
    df_pessoas_silver.where(F.col("tipo_entidade") == "Produtora")
    .join(filmes_conhecidos, "id_filme", "left_semi"),
    bc.join(dm.select("sk_movie_id", "id_filme"), "sk_movie_id").join(dc, "sk_company_id")
    .select("id_filme", F.col("nome_produtora").alias("nome_entidade")),
    ["id_filme", "nome_entidade"], TABELA_PONTE_PRODUTORAS,
)

# campos opcionais podem faltar na fonte, mas não podem apagar o documento para a ia
if contexto.where(
    F.col("title").isNull() | (F.trim("title") == "")
    | F.col("llm_context_document").isNull() | (F.trim("llm_context_document") == "")
).limit(1).count():
    raise AssertionError("gold.gold_genai_movies_context: título ou documento vazio")
if dp.where(~F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")).limit(1).count():
    raise AssertionError("gold.dim_people: tipo de pessoa inválido")

display(spark.createDataFrame(
    [(nome, df.count()) for nome, df in tabelas_gold.items()],
    ["tabela_gold", "quantidade_registros"],
))
print("Validações da Gold concluídas; consulte também os avisos de vínculos ausentes na Silver")


In [ ]:
# consultas analíticas da camada gold
# mantém o id natural no modelo e usa a obra canônica quando a pergunta mede obras distintas

from pyspark.sql.window import Window

df_analytics_movies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
df_analytics_fact = spark.table(f"{SCHEMA_DESTINO}.{TABELA_FATO_PERFORMANCE}")
df_analytics_people_bridge = spark.table(f"{SCHEMA_DESTINO}.{TABELA_PONTE_PESSOAS}")
df_analytics_company_bridge = spark.table(f"{SCHEMA_DESTINO}.{TABELA_PONTE_PRODUTORAS}")
df_analytics_people = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PESSOAS}")
df_analytics_companies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PRODUTORAS}")
df_analytics_genres_bridge = spark.table(f"{SCHEMA_DESTINO}.{TABELA_PONTE_GENEROS}")
df_analytics_genres = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")

# a referência vem do último lançamento realizado na base, como pede o enunciado
data_referencia = (
    df_analytics_movies
    .where(
        (F.col("status_filme") == "Lançado")
        & F.col("data_lancamento").isNotNull()
        & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(F.max("data_lancamento").alias("data_referencia"))
    .first()["data_referencia"]
)
if data_referencia is None:
    raise ValueError("Não há lançamento realizado para definir a referência das análises")
print(f"Data de referência das perguntas 5 e 6: {data_referencia}")

# pergunta um: soma as receitas conhecidas dos filmes lançados sem juntar tabelas-ponte
df_receita_total = df_analytics_fact.agg(
    F.round(F.sum("receita_brl"), 2).alias("receita_total_brl")
)
display(df_receita_total)

# pergunta dois: desempata pela chave do filme para repetir os mesmos cinco resultados
df_top_popularidade = (
    df_analytics_movies.select("sk_movie_id", "titulo")
    .join(df_analytics_fact.select("sk_movie_id", "popularidade"), "sk_movie_id")
    .where(F.col("popularidade").isNotNull())
    .orderBy(F.col("popularidade").desc(), F.col("sk_movie_id").asc()).limit(5)
)
display(df_top_popularidade)

# pergunta três: cada id natural conta uma vez em cada gênero, incluindo o catálogo completo
df_filmes_por_genero = (
    df_analytics_genres_bridge
    .join(df_analytics_genres, "sk_genre_id")
    .groupBy("sk_genre_id", "nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("quantidade_filmes"))
    .orderBy(F.col("quantidade_filmes").desc(), F.col("nome_genero").asc())
)
display(df_filmes_por_genero)

# pergunta quatro: rank considera somente a receita e conserva empates na décima posição
# o sk_movie_id apenas ordena os empatados de forma determinística e não altera o ranking
df_top_receita = (
    df_analytics_movies.select("sk_movie_id", "titulo")
    .join(df_analytics_fact.select("sk_movie_id", "receita_usd", "receita_brl"), "sk_movie_id")
    .where(F.col("receita_usd").isNotNull())
    .withColumn("ranking_receita", F.rank().over(Window.orderBy(F.col("receita_usd").desc())))
    .where(F.col("ranking_receita") <= 10)
    .select("ranking_receita", "sk_movie_id", "titulo", "receita_usd", "receita_brl")
    .orderBy("ranking_receita", "sk_movie_id")
)
display(df_top_receita)

# pergunta cinco: apresenta duas leituras para tornar explícito o grão da contagem
# a resposta principal segue o grão físico pedido na atividade e conta cada sk_movie_id
# a análise complementar agrupa ids diferentes da mesma obra por id_obra_canonica
df_filmes_ultimos_dois_anos = df_analytics_movies.where(
    (F.col("status_filme") == "Lançado")
    & F.col("data_lancamento").between(
        F.add_months(F.lit(data_referencia), -24), F.lit(data_referencia)
    )
).select("sk_movie_id", "id_obra_canonica")
# reúne uma vez as participações de atores no recorte para aplicar as duas métricas
df_participacoes_atores_dois_anos = (
    df_analytics_people_bridge
    .join(df_analytics_people.where(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .join(df_filmes_ultimos_dois_anos, "sk_movie_id")
    .select("sk_person_id", "nome_pessoa", "sk_movie_id", "id_obra_canonica")
    .dropDuplicates()
)

# resposta principal: quantidade de filmes conforme o identificador do modelo Gold
# o agrupamento usa o identificador porque nomes podem variar ou não existir de forma confiável na origem
df_top_ator_por_filmes = (
    df_participacoes_atores_dois_anos
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("quantidade_filmes"))
    .withColumn("ranking_participacao", F.rank().over(Window.orderBy(F.col("quantidade_filmes").desc())))
    .where(F.col("ranking_participacao") == 1)
    .select("ranking_participacao", "sk_person_id", "nome_pessoa", "quantidade_filmes")
    .orderBy("nome_pessoa", "sk_person_id")
)
print("Pergunta 5 - resposta principal por filmes (sk_movie_id)")
display(df_top_ator_por_filmes)

# análise complementar: quantidade de obras cinematográficas após consolidar ids equivalentes
df_top_ator_por_obras_canonicas = (
    df_participacoes_atores_dois_anos
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(F.countDistinct("id_obra_canonica").alias("quantidade_obras_canonicas"))
    .withColumn(
        "ranking_participacao",
        F.rank().over(Window.orderBy(F.col("quantidade_obras_canonicas").desc())),
    )
    .where(F.col("ranking_participacao") == 1)
    .select(
        "ranking_participacao", "sk_person_id", "nome_pessoa",
        "quantidade_obras_canonicas",
    )
    .orderBy("nome_pessoa", "sk_person_id")
)
print("Pergunta 5 - análise complementar por obras canônicas (id_obra_canonica)")
display(df_top_ator_por_obras_canonicas)

# pergunta seis: filtra os lançamentos antes da soma e agrega uma vez por obra e produtora
# o lucro integral é atribuído a cada produtora associada, pois a fonte não informa cotas de participação
df_filmes_ultimos_cinco_anos = df_analytics_movies.where(
    (F.col("status_filme") == "Lançado")
    & F.col("data_lancamento").between(
        F.add_months(F.lit(data_referencia), -60), F.lit(data_referencia)
    )
).select("sk_movie_id", "id_obra_canonica")
# consolida primeiro as cópias da mesma obra para não somar o lucro repetidamente
df_lucro_por_obra = (
    df_analytics_company_bridge
    .select("sk_movie_id", "sk_company_id").dropDuplicates()
    .join(df_filmes_ultimos_cinco_anos, "sk_movie_id")
    .join(df_analytics_fact.select("sk_movie_id", "lucro_usd", "lucro_brl"), "sk_movie_id")
    .join(df_analytics_companies, "sk_company_id")
    .groupBy("sk_company_id", "nome_produtora", "id_obra_canonica")
    .agg(
        F.max("lucro_usd").alias("lucro_usd"),
        F.max("lucro_brl").alias("lucro_brl"),
    )
)
df_top_produtora_lucro = (
    df_lucro_por_obra
    .groupBy("sk_company_id", "nome_produtora")
    .agg(
        F.round(F.sum("lucro_usd"), 2).alias("lucro_total_usd"),
        F.round(F.sum("lucro_brl"), 2).alias("lucro_total_brl"),
        F.count("lucro_brl").alias("filmes_com_lucro_conhecido"),
        F.sum(F.when(F.col("lucro_brl").isNull(), 1).otherwise(0)).alias("filmes_sem_lucro_informado"),
    )
    .where(F.col("lucro_total_brl").isNotNull())
    .withColumn("ranking_lucro", F.rank().over(Window.orderBy(F.col("lucro_total_brl").desc())))
    .where(F.col("ranking_lucro") == 1)
    .orderBy("nome_produtora", "sk_company_id")
)
display(df_top_produtora_lucro)
